In [6]:
import json
import pandas as pd
from datetime import datetime, timezone
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [ ]:
# connect to chrome instance with remote debugging enabled
options = Options()
options.add_experimental_option('debuggerAddress', '127.0.0.1:9222')

driver = webdriver.Chrome(options=options)
print(f"Connected to: {driver.current_url}")

Connected to: https://starlink.com/account/service-line/AST-2293597-46342-54?selectedDevice=ut01000000-00000000-0060d786&page=0&limit=5


In [ ]:
raw_data = driver.execute_async_script("""
    const callback = arguments[arguments.length - 1];
    fetch('https://starlink.com/api/telemetryagg/v1/data-usage/account/ACC-2735603-74738-20/service-line/AST-2293597-46342-54/annotated', {4
                                       
        credentials: 'include'
    })
                                       
    .then(r => r.json())
    .then(data => callback(data))
    .catch(err => callback({error: err.toString()}));
""")

if 'error' in raw_data:
    print(f"Fetch failed: {raw_data['error']}")
else:
    print("Data fetched successfully")
    print(f"Keys: {list(raw_data.keys())}")

Data fetched successfully
Keys: ['content', 'errors', 'information', 'isValid', 'warnings']


In [ ]:
# OPTIONAL: Load from existing JSON file instead of scraping
# Uncomment the lines below to use a saved JSON file

# with open('data/starlink_data.json', 'r') as f:
#     raw_data = json.load(f)
# 
# billing_cycles = raw_data['content']['billingCyclesAnnotated']
# print(f"Loaded {len(billing_cycles)} billing cycles from JSON")
# print(f"Keys: {list(raw_data.keys())}")

In [9]:
# Save raw JSON locally as backup
with open('data/starlink_data.json', 'w') as f:
    json.dump(raw_data, f, indent=2)

print("Raw JSON saved to starlink_data.json")

Raw JSON saved to starlink_data.json


In [10]:
# Parse billing cycles into a DataFrame
billing_cycles = raw_data['content']['billingCyclesAnnotated']
now = datetime.now(timezone.utc).replace(tzinfo=None)
rows = []

for cycle in billing_cycles:
    start_date = pd.to_datetime(cycle['startDate'].split('T')[0])
    for i, usage_array in enumerate(cycle['dailyData']):
        if not usage_array:
            continue
        date = start_date + pd.Timedelta(days=i)
        if date > now:
            continue
        rows.append({'date': date, 'usage_gb': round(usage_array[0], 2)})

df = pd.DataFrame(rows)
df['date'] = pd.to_datetime(df['date'])
df['day_of_week'] = df['date'].dt.day_name()
df['week'] = df['date'].dt.strftime('%G-W%V')
df['month'] = df['date'].dt.strftime('%Y-%m')

print(f"Processed {len(df)} days of data")
df.head()

Processed 186 days of data


,date,usage_gb,day_of_week,week,month
0,2025-11-17,17.49,Monday,2025-W47,2025-11
1,2025-11-18,13.31,Tuesday,2025-W47,2025-11
2,2025-11-19,6.95,Wednesday,2025-W47,2025-11
3,2025-11-20,12.14,Thursday,2025-W47,2025-11
4,2025-11-21,10.10,Friday,2025-W47,2025-11


In [11]:
# Overall stats
stats = df['usage_gb'].agg(['count', 'sum', 'mean']).round(2)
stats.index = ['Total Days', 'Total GB', 'Avg Daily GB']
print(stats)

Total Days       186.00
Total GB        2345.72
Avg Daily GB      12.61
Name: usage_gb, dtype: float64


In [12]:
# Top 5 highest usage days
df.nlargest(5, 'usage_gb')[['date', 'day_of_week', 'usage_gb']]

,date,day_of_week,usage_gb
181,2026-05-17,Sunday,45.09
27,2025-12-14,Sunday,44.74
43,2025-12-30,Tuesday,35.42
47,2026-01-03,Saturday,33.69
28,2025-12-15,Monday,33.21


In [13]:
# Average usage by day of week
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_avg = df.groupby('day_of_week')['usage_gb'].mean().round(2).reindex(dow_order)
print(dow_avg)

day_of_week
Monday       13.00
Tuesday      11.98
Wednesday    12.36
Thursday     10.82
Friday       12.26
Saturday     13.24
Sunday       14.71
Name: usage_gb, dtype: float64


In [14]:
# Weekly and monthly summaries
weekly = df.groupby('week')['usage_gb'].sum().round(2).reset_index()
weekly.columns = ['week', 'total_gb']

monthly = df.groupby('month')['usage_gb'].agg(
    total_gb='sum',
    average_daily_gb='mean',
    max_daily_gb='max',
    min_daily_gb='min',
    days_count='count'
).round(2).reset_index()

display(weekly)
display(monthly)

,week,total_gb
0,2025-W47,83.90
1,2025-W48,118.23
2,2025-W49,107.63
3,2025-W50,129.19
4,2025-W51,137.36
5,2025-W52,137.33
6,2026-W01,171.94
7,2026-W02,108.16
8,2026-W03,53.14
9,2026-W04,69.69


,month,total_gb,average_daily_gb,max_daily_gb,min_daily_gb,days_count
0,2025-11,202.13,14.44,23.19,6.61,14
1,2025-12,590.22,19.04,44.74,9.75,31
2,2026-01,380.36,12.27,33.69,2.51,31
3,2026-02,120.77,4.31,9.12,0.88,28
4,2026-03,197.23,6.36,15.59,0.90,31
5,2026-04,517.98,17.27,33.01,4.61,30
6,2026-05,337.03,16.05,45.09,4.95,21


In [18]:
# Export raw scraped data to CSV
raw_df = pd.DataFrame([{
    'date': pd.to_datetime(cycle['startDate'].split('T')[0]) + pd.Timedelta(days=i),
    'usage_gb': usage_array[0] if usage_array else None
} for cycle in billing_cycles 
  for i, usage_array in enumerate(cycle['dailyData']) 
  if usage_array], columns=['date', 'usage_gb'])

raw_df.to_csv('data/starlink_raw_export.csv', index=False)
print(f"Raw data exported: {len(raw_df)} records to data/starlink_raw_export.csv")

Raw data exported: 212 records to data/starlink_raw_export.csv


In [15]:
# Export CSVs
df[['date', 'day_of_week', 'week', 'month', 'usage_gb']].to_csv('data/starlink_daily_usage.csv', index=False)
weekly.to_csv('data/starlink_weekly_summary.csv', index=False)
monthly.to_csv('data/starlink_monthly_summary.csv', index=False)
print("CSVs saved.")

CSVs saved.


In [16]:
# Write report
top_days = df.nlargest(5, 'usage_gb')

with open('data/starlink_usage_report.txt', 'w', encoding='utf-8') as f:
    f.write('=' * 60 + '\n')
    f.write('STARLINK DATA USAGE ANALYSIS REPORT\n')
    f.write('=' * 60 + '\n\n')

    f.write('OVERALL STATISTICS\n')
    f.write('-' * 30 + '\n')
    for label, val in stats.items():
        f.write(f"{label:<20}: {val}\n")

    f.write('\nTOP 5 HIGHEST USAGE DAYS\n')
    f.write('-' * 30 + '\n')
    for i, (_, row) in enumerate(top_days.iterrows(), 1):
        f.write(f"{i}. {row['date'].strftime('%Y-%m-%d')} ({row['day_of_week']}): {row['usage_gb']} GB\n")

    f.write('\nAVERAGE USAGE BY DAY OF WEEK\n')
    f.write('-' * 30 + '\n')
    for day, avg in dow_avg.items():
        f.write(f"{day:<10}: {avg} GB\n")

    f.write('\n' + '=' * 60 + '\n')
    f.write('MONTHLY DATA USAGE REPORT\n')
    f.write('=' * 60 + '\n')
    for _, row in monthly.iterrows():
        f.write(f"\n{row['month']}\n")
        f.write(f"  Total Usage:    {row['total_gb']} GB\n")
        f.write(f"  Daily Average:  {row['average_daily_gb']} GB\n")
        f.write(f"  Peak Day:       {row['max_daily_gb']} GB\n")
        f.write(f"  Lowest Day:     {row['min_daily_gb']} GB\n")
        f.write(f"  Days Recorded:  {int(row['days_count'])}\n")
    f.write('\n' + '=' * 60 + '\n')
    f.write(f"Grand Total: {monthly['total_gb'].sum().round(2)} GB across {len(monthly)} months\n")
    f.write('\n' + '=' * 60 + '\n')
    f.write(f"Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

print("Report saved: starlink_usage_report.txt")

Report saved: starlink_usage_report.txt


In [17]:
driver.quit()
print("Browser connection closed.")

Browser connection closed.
